# In Silico Saturation Mutagenesis → HDF5
For each position in a region, substitute every non-reference base and record the change in model predictions.

In [ ]:
import sys
sys.path.insert(0, '../../..')

import numpy as np
import torch
import h5py
from pathlib import Path
from scipy.special import rel_entr

from src.infer._utils import load_model, parse_bed, parse_region, fetch_one_hot, rc_average
from src.data.data_utils import GenomeSequence

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
CHECKPOINT     = 'outputs/basenji2_wt/checkpoints/best_model.pt'
CONFIG         = 'src/configs/transformer_wt.yaml'
FASTA          = '/path/to/genome.fa'
SPECIES        = 'rice'        # must match a name in manifest.yaml

# Provide either REGION (single) or BED (multiple)
REGION         = 'chr01:1000000-1001000'   # set to None to use BED
BED            = None                       # e.g. 'data/regions.bed'

# Length of center region to mutate (bp). None = mutate full window.
MUT_LEN        = 200

# Stats to compute. Subset of: 'sum', 'center', 'scd', 'js'
#   sum    – sum over bins, mean-centered across bases (default basenji stat)
#   center – sum over center 2 bins, mean-centered across bases
#   scd    – sqrt(Σ(pred_mut - pred_ref)² over bins), no centering
#   js     – symmetric KL divergence across bins
SAD_STATS      = ['sum', 'scd']

# Ensemble forward + reverse-complement predictions
RC             = False

MUT_BATCH_SIZE = 64
OUTPUT         = 'output/ism_scores.h5'

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, cfg = load_model(CHECKPOINT, CONFIG, device)
window_size = cfg['data']['input_window_length']
genome = GenomeSequence(FASTA)

# probe output bin count
with torch.no_grad():
    _x = torch.zeros(1, 4, window_size, device=device)
    num_bins = model(_x, head=SPECIES)['rt_signals'].shape[1]
num_phases = cfg['model']['n_output_bins']
print(f'Model loaded on {device} | output: {num_bins} bins × {num_phases} phases')

if BED:
    regions = list(parse_bed(BED))
else:
    chrom, start, end = parse_region(REGION)
    regions = [(chrom, start, end, REGION, '+')]
print(f'{len(regions)} region(s)')

In [ ]:
def _predict(x):
    """Run model (with optional RC ensemble). x: [B, 4, L] tensor → [B, T, P] numpy."""
    if RC:
        return rc_average(model, x, SPECIES).cpu().numpy()
    with torch.no_grad():
        return model(x, head=SPECIES)['rt_signals'].cpu().numpy()


def ism_region(one_hot_np, mut_start, mut_end):
    """
    Compute SAD stats for every alt-base substitution in [mut_start, mut_end).

    Returns
    -------
    ref_pred : [T, P]  reference prediction (bins × phases)
    scores   : dict stat → [mut_len, 4, P]  per-position, per-base, per-phase
    """
    mut_len = mut_end - mut_start
    T = num_bins

    # center bin range (matches basenji logic)
    center_start = T // 2
    center_end   = center_start + (2 if T % 2 == 0 else 1)

    # reference prediction
    ref_t    = torch.tensor(one_hot_np[None], dtype=torch.float32).to(device)
    ref_pred = _predict(ref_t)[0]                          # [T, P]
    ref_pred_norm = ref_pred / (ref_pred.sum(axis=0) + 1e-8)  # for JS

    # initialize score arrays: (mut_len, 4 bases, P phases)
    scores = {s: np.zeros((mut_len, 4, num_phases), dtype=np.float32) for s in SAD_STATS}

    # fill reference-base slots for sum / center (ref SCD / JS = 0 by definition)
    ref_sum    = ref_pred.sum(axis=0)                          # [P]
    ref_center = ref_pred[center_start:center_end].sum(axis=0) # [P]
    for li in range(mut_len):
        col = one_hot_np[:, mut_start + li]
        if col.sum() == 0:
            continue
        rb = col.argmax()
        if 'sum'    in SAD_STATS: scores['sum'][li, rb]    = ref_sum
        if 'center' in SAD_STATS: scores['center'][li, rb] = ref_center

    # build alt-mutation list
    mut_list = []
    for i in range(mut_start, mut_end):
        col = one_hot_np[:, i]
        rb  = col.argmax() if col.sum() > 0 else -1
        for b in range(4):
            if b == rb:
                continue
            mut = one_hot_np.copy()
            mut[:, i] = 0.0
            mut[b, i] = 1.0
            mut_list.append((i - mut_start, b, mut))

    # batch inference
    for bs in range(0, len(mut_list), MUT_BATCH_SIZE):
        batch = mut_list[bs: bs + MUT_BATCH_SIZE]
        x     = torch.tensor(np.stack([m[2] for m in batch]),
                              dtype=torch.float32).to(device)
        preds = _predict(x)                                    # [B, T, P]

        for k, (li, b, _) in enumerate(batch):
            p     = preds[k]                                   # [T, P]
            if 'sum'    in SAD_STATS:
                scores['sum'][li, b]    = p.sum(axis=0)
            if 'center' in SAD_STATS:
                scores['center'][li, b] = p[center_start:center_end].sum(axis=0)
            if 'scd'    in SAD_STATS:
                scores['scd'][li, b]    = np.sqrt(((p - ref_pred) ** 2).sum(axis=0))
            if 'js'     in SAD_STATS:
                p_norm = p / (p.sum(axis=0) + 1e-8)
                ref_alt = rel_entr(ref_pred_norm, p_norm).sum(axis=0)
                alt_ref = rel_entr(p_norm, ref_pred_norm).sum(axis=0)
                scores['js'][li, b]     = (ref_alt + alt_ref) / 2

    # mean-center sum / center across the 4 bases (axis=1)
    for stat in ('sum', 'center'):
        if stat in SAD_STATS:
            scores[stat] -= scores[stat].mean(axis=1, keepdims=True)

    return ref_pred, scores

In [ ]:
n = len(regions)
seq_mid = window_size // 2
if MUT_LEN is None:
    mut_start, mut_end = 0, window_size
else:
    mut_up    = MUT_LEN // 2
    mut_start = seq_mid - mut_up
    mut_end   = mut_start + MUT_LEN
mut_len = mut_end - mut_start

Path(OUTPUT).parent.mkdir(parents=True, exist_ok=True)

with h5py.File(OUTPUT, 'w') as hf:
    hf.attrs['mut_start'] = mut_start
    hf.attrs['mut_end']   = mut_end
    hf.attrs['sad_stats'] = SAD_STATS
    hf.create_dataset('seqnames', data=np.array([r[3] for r in regions], dtype='S'))
    hf.create_dataset('chrom',    data=np.array([r[0] for r in regions], dtype='S'))
    hf.create_dataset('start',    data=np.array([r[1] for r in regions], dtype=np.int32))
    hf.create_dataset('end',      data=np.array([r[2] for r in regions], dtype=np.int32))
    hf.create_dataset('strand',   data=np.array([r[4] for r in regions], dtype='S'))
    # seqs: full window one-hot for context
    ds_seqs = hf.create_dataset('seqs',     shape=(n, 4, window_size),          dtype=np.float32)
    # ref_pred: full bin × phase prediction (not reduced)
    ds_ref  = hf.create_dataset('ref_pred', shape=(n, num_bins, num_phases),     dtype=np.float32)
    # one dataset per stat: (n, mut_len, 4 bases, P phases)
    ds_stat = {s: hf.create_dataset(s, shape=(n, mut_len, 4, num_phases), dtype=np.float32)
               for s in SAD_STATS}

    for idx, (chrom, start, end, name, strand) in enumerate(regions):
        print(f'[{idx+1}/{n}] {name}')
        oh = fetch_one_hot(genome, chrom, start, end, window_size, strand)
        ref_pred, scores = ism_region(oh, mut_start, mut_end)
        ds_seqs[idx] = oh
        ds_ref[idx]  = ref_pred
        for s in SAD_STATS:
            ds_stat[s][idx] = scores[s]

print(f'Written to {OUTPUT}')

## Visualization

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches
import numpy as np
import h5py
from pathlib import Path

PHASE_NAMES = ['G1', 'ES', 'MS', 'LS']
BASE_NAMES  = ['A', 'C', 'G', 'T']
_COLORS     = ['red', 'blue', 'orange', 'green']  # A C G T

LOGO_STAT  = 'sum' if 'sum' in SAD_STATS else SAD_STATS[0]
HEAT_STAT  = 'scd' if 'scd' in SAD_STATS else SAD_STATS[0]
PLOT_PHASE = 1   # 0=G1  1=ES  2=MS  3=LS

PDF_DIR = Path(OUTPUT).parent / 'plots'
PDF_DIR.mkdir(parents=True, exist_ok=True)

# ── load all regions and compute per-region delta ─────────────────────────────
with h5py.File(OUTPUT, 'r') as hf:
    ms       = hf.attrs['mut_start']
    me       = hf.attrs['mut_end']
    n_seqs   = hf['seqs'].shape[0]
    all_seqs = hf['seqs'][:, :, ms:me].transpose(0, 2, 1)   # [N, mut_len, 4]
    all_logo = hf[LOGO_STAT][:, :, :, PLOT_PHASE]            # [N, mut_len, 4]
    all_heat = hf[HEAT_STAT][:, :, :, PLOT_PHASE]            # [N, mut_len, 4]

mut_len = all_seqs.shape[1]

ref_s_all  = all_logo[np.arange(n_seqs)[:, None],
                      np.arange(mut_len)[None, :],
                      all_seqs.argmax(axis=2)]               # [N, mut_len]
delta_all  = all_logo - ref_s_all[:, :, None]                # [N, mut_len, 4]

loss_mean  = (-delta_all.min(axis=2)).mean(axis=0)
gain_mean  = ( delta_all.max(axis=2)).mean(axis=0)
heat_mean  = all_heat.mean(axis=0)
delta_mean = delta_all.mean(axis=0)
freq_mean  = all_seqs.mean(axis=0)

print(f'Loaded {n_seqs} regions, mut_len={mut_len}')
print(f'PDFs will be saved to: {PDF_DIR}')

In [ ]:
def _plot_letter(ax, base_idx, left, height, base_y=0):
    c = _COLORS[base_idx]
    if base_idx == 0:   # A
        for poly in [
            np.array([[0.0,0.0],[0.5,1.0],[0.5,0.8],[0.2,0.0]]),
            np.array([[1.0,0.0],[0.5,1.0],[0.5,0.8],[0.8,0.0]]),
            np.array([[0.225,0.45],[0.775,0.45],[0.85,0.3],[0.15,0.3]])]:
            ax.add_patch(matplotlib.patches.Polygon(
                poly * np.array([1, height]) + np.array([left, base_y]),
                facecolor=c, edgecolor=c))
    elif base_idx == 1:  # C
        ax.add_patch(matplotlib.patches.Ellipse(
            xy=[left+0.65, base_y+0.5*height], width=1.3, height=height, facecolor=c, edgecolor=c))
        ax.add_patch(matplotlib.patches.Ellipse(
            xy=[left+0.65, base_y+0.5*height], width=0.91, height=0.7*height, facecolor='white', edgecolor='white'))
        ax.add_patch(matplotlib.patches.Rectangle(
            xy=[left+1, base_y], width=1.0, height=height, facecolor='white', edgecolor='white'))
    elif base_idx == 2:  # G
        ax.add_patch(matplotlib.patches.Ellipse(
            xy=[left+0.65, base_y+0.5*height], width=1.3, height=height, facecolor=c, edgecolor=c))
        ax.add_patch(matplotlib.patches.Ellipse(
            xy=[left+0.65, base_y+0.5*height], width=0.91, height=0.7*height, facecolor='white', edgecolor='white'))
        ax.add_patch(matplotlib.patches.Rectangle(
            xy=[left+1, base_y], width=1.0, height=height, facecolor='white', edgecolor='white'))
        ax.add_patch(matplotlib.patches.Rectangle(
            xy=[left+0.825, base_y+0.085*height], width=0.174, height=0.415*height, facecolor=c, edgecolor=c))
        ax.add_patch(matplotlib.patches.Rectangle(
            xy=[left+0.625, base_y+0.35*height], width=0.374, height=0.15*height, facecolor=c, edgecolor=c))
    else:                # T
        ax.add_patch(matplotlib.patches.Rectangle(
            xy=[left+0.4, base_y], width=0.2, height=height, facecolor=c, edgecolor=c))
        ax.add_patch(matplotlib.patches.Rectangle(
            xy=[left, base_y+0.8*height], width=1.0, height=0.2*height, facecolor=c, edgecolor=c))


def plot_seqlogo(ax, seq_1hot, scores_4l, pseudo_pct=0.05):
    """Loss logo: letter height = how much signal is lost when this base is mutated away."""
    ref_s = scores_4l[np.arange(len(seq_1hot)), seq_1hot.argmax(axis=1)]
    delta = scores_4l - ref_s[:, None]
    loss  = -delta.min(axis=1)
    loss_cp = loss + pseudo_pct * loss.max()
    logo_4l = seq_1hot * loss_cp[:, None]

    max_h = 0
    for li in range(len(seq_1hot)):
        cur_h = 0
        for score, ni in sorted([(logo_4l[li, ni], ni) for ni in range(4)]):
            if score > 0:
                _plot_letter(ax, ni, li, score, base_y=cur_h)
                cur_h += score
        max_h = max(max_h, cur_h)

    ax.set_xlim(0, len(seq_1hot))
    ax.set_ylim(-0.05 * max_h, max_h * 1.05)
    ax.set_yticks([])

In [ ]:
# ── Average loss logo → PDF ───────────────────────────────────────────────────
loss_all      = (-delta_all.min(axis=2))
loss_mean_pos = loss_all.mean(axis=0)
loss_cp       = loss_mean_pos + 0.05 * loss_mean_pos.max()
logo_4l       = freq_mean * loss_cp[:, None]

fig, ax = plt.subplots(figsize=(20, 2.5))
max_h = 0
for li in range(mut_len):
    cur_h = 0
    for score, ni in sorted([(logo_4l[li, ni], ni) for ni in range(4)]):
        if score > 0:
            _plot_letter(ax, ni, li, score, base_y=cur_h)
            cur_h += score
    max_h = max(max_h, cur_h)

ax.set_xlim(0, mut_len)
ax.set_ylim(-0.05 * max(max_h, 1e-6), max(max_h, 1e-6) * 1.1)
ax.set_yticks([])
ax.set_ylabel('Loss', fontsize=9)
ax.set_xlabel('Position (within mut region)')
ax.set_title(f'Average loss logo  ({n_seqs} regions)  |  phase: {PHASE_NAMES[PLOT_PHASE]}  |  stat: {LOGO_STAT}', fontsize=10)
plt.tight_layout()
out = PDF_DIR / f'logo_{PHASE_NAMES[PLOT_PHASE]}.pdf'
fig.savefig(out, bbox_inches='tight')
plt.close()
print(f'Saved: {out}')

In [ ]:
# ── Average loss / gain line → PDF ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(20, 2))
rdbu = plt.cm.RdBu_r(np.linspace(0, 1, 10))
ax.plot(loss_mean, color=rdbu[0],  linewidth=0.8, label='loss')
ax.plot(gain_mean, color=rdbu[-1], linewidth=0.8, label='gain')
ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.set_xlim(0, mut_len)
ax.legend(fontsize=8)
ax.set_ylabel('SAD', fontsize=9)
ax.set_xlabel('Position (within mut region)')
ax.set_title(f'Average loss / gain  ({n_seqs} regions)  |  phase: {PHASE_NAMES[PLOT_PHASE]}  |  stat: {LOGO_STAT}', fontsize=10)
plt.tight_layout()
out = PDF_DIR / f'loss_gain_{PHASE_NAMES[PLOT_PHASE]}.pdf'
fig.savefig(out, bbox_inches='tight')
plt.close()
print(f'Saved: {out}')

In [ ]:
# ── Average heatmap → PDF ─────────────────────────────────────────────────────
mat  = heat_mean.T
vmax = max(0.05, np.percentile(np.abs(mat), 98))

fig, ax = plt.subplots(figsize=(20, 1.5))
im = ax.imshow(mat, aspect='auto', cmap='RdBu_r',
               vmin=-vmax, vmax=vmax, interpolation='nearest')
ax.set_yticks(range(4))
ax.set_yticklabels(BASE_NAMES, fontsize=8)
ax.set_xlabel('Position (within mut region)')
ax.set_title(f'Average heatmap  ({n_seqs} regions)  |  phase: {PHASE_NAMES[PLOT_PHASE]}  |  stat: {HEAT_STAT}', fontsize=10)
plt.colorbar(im, ax=ax, fraction=0.015, pad=0.01)
plt.tight_layout()
out = PDF_DIR / f'heatmap_{PHASE_NAMES[PLOT_PHASE]}.pdf'
fig.savefig(out, bbox_inches='tight')
plt.close()
print(f'Saved: {out}')